# Q8 - Churn predictor

Loads `churn_model.joblib` (notebook 07), lets you **enter a player's aggregated features**, shows the **prediction**, and **stores every prediction** in `predictions_log.csv` so you can review them later. A **SHAP diagram** explains each prediction (which features push the churn risk up or down), and a second tool scores a whole file of players at once.

How to use: run the cells, fill in the form of section 2, click **Predict and save**. Every field starts at the median of the training players: change the ones you know and leave the others (an unknown value is then treated as a typical player).

Reading the result: `P(still active)` is the model's probability, `churn risk = 1 - P(still active)`. A player is flagged **at risk** when the churn risk is above the threshold of the model. The model is moderate (about 0.8 ROC-AUC on 840 players, noisy label): use it to rank players, not as a certainty.

In [2]:
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as w
import shap
from IPython.display import clear_output, display

FEATURES_PATH: Path = Path('player_features.parquet')  # only used for the global SHAP view
MODEL_PATH: Path = Path('churn_model.joblib')
LOG_PATH: Path = Path('predictions_log.csv')

artifact: dict = joblib.load(MODEL_PATH)
MODEL = artifact['model']
FEATURES: list[str] = artifact['features']
FEATURE_INFO: dict = artifact['feature_info']
DEFAULTS: dict = artifact['defaults']
RANGES: dict = artifact['ranges']
THRESHOLD: float = artifact['threshold_churn_risk']
print(f"model trained on {artifact['trained_on']['players']} players ({artifact['trained_on']['date']}) | churn-risk threshold {THRESHOLD:.2f}")
print('metrics:', {k: v for k, v in artifact['metrics'].items()})
print(artifact['note'])

FileNotFoundError: [Errno 2] No such file or directory: 'churn_model.joblib'

## 1. Prediction and storage functions

`predict_players` works on a table (one row per player); `predict_one` and the form call it. `save_predictions` appends to the log file.

In [ ]:
def predict_players(df: pd.DataFrame) -> pd.DataFrame:
    # df: columns = the model features (missing columns / values are allowed: the model imputes the training median)
    X = df.reindex(columns=FEATURES).astype(float)
    p_active = MODEL.predict_proba(X)[:, 1]
    out = df.copy()
    out['p_active'] = p_active.round(3)
    out['churn_risk'] = (1 - p_active).round(3)
    out['prediction'] = np.where(out['churn_risk'] >= THRESHOLD, 'At risk (likely churn)', 'Likely still active')
    return out


def save_predictions(df: pd.DataFrame) -> None:
    df = df.copy()
    df.insert(0, 'timestamp', datetime.now().isoformat(timespec='seconds'))
    df.to_csv(LOG_PATH, mode='a', header=not LOG_PATH.exists(), index=False)


def load_log() -> pd.DataFrame:
    return pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()


def predict_one(player_id: str, values: dict, store: bool = True) -> pd.Series:
    row = pd.DataFrame([{'player_id': player_id, **{f: values.get(f, np.nan) for f in FEATURES}}])
    res = predict_players(row)
    if store:
        save_predictions(res)
    return res.iloc[0]


# SHAP: contribution of every feature to the churn risk of a player (TreeExplainer on the random forest)
IMPUTER, FOREST = MODEL.steps[0][1], MODEL.steps[-1][1]
EXPLAINER = shap.TreeExplainer(FOREST)
LABELS: list[str] = [FEATURE_INFO[f][0] for f in FEATURES]
CHURN_CLASS: int = list(FOREST.classes_).index(0)  # still_active = False


def shap_churn(df: pd.DataFrame) -> shap.Explanation:
    # df: columns = the model features; missing values are imputed with the training median, as in the model
    X = pd.DataFrame(IMPUTER.transform(df.reindex(columns=FEATURES).astype(float)), columns=FEATURES, index=df.index)
    sv = EXPLAINER(X)
    return shap.Explanation(values=sv.values[:, :, CHURN_CLASS], base_values=sv.base_values[:, CHURN_CLASS], data=X.values, feature_names=LABELS)


def plot_waterfall(df_row: pd.DataFrame, title: str) -> None:
    # red bars push the churn risk up, blue bars push it down; start = average churn risk of the training players
    shap.plots.waterfall(shap_churn(df_row)[0], max_display=12, show=False)
    plt.title(title, loc='left')
    plt.tight_layout()
    plt.show()

## 2. Form: enter a player, predict, save

In [ ]:
player_id = w.Text(value='', description='Player id', placeholder='any name or id', style={'description_width': '220px'}, layout=w.Layout(width='520px'))
inputs: dict[str, w.FloatText] = {}
rows = [player_id]
for f in FEATURES:
    label, unit = FEATURE_INFO[f]
    lo, hi = RANGES[f]
    inputs[f] = w.FloatText(value=round(DEFAULTS[f], 3), description=label, style={'description_width': '220px'}, layout=w.Layout(width='520px'))
    rows.append(w.HBox([inputs[f], w.HTML(f"<small>&nbsp;{unit} | training range {lo:.3g} - {hi:.3g}</small>")]))
use_default = w.Checkbox(value=False, description='Reset all fields to the median player')
button = w.Button(description='Predict and save', button_style='primary', icon='check')
out = w.Output()


def on_reset(change):
    if change['new']:
        for f in FEATURES:
            inputs[f].value = round(DEFAULTS[f], 3)
        use_default.value = False


def on_click(_):
    with out:
        clear_output()
        pid = player_id.value.strip() or f"player_{datetime.now().strftime('%H%M%S')}"
        res = predict_one(pid, {f: inputs[f].value for f in FEATURES})
        colour = '#c0392b' if res['churn_risk'] >= THRESHOLD else '#27ae60'
        display(w.HTML(
            f"<div style='border:2px solid {colour};padding:10px;width:520px'>"
            f"<b>{pid}</b><br><span style='font-size:20px;color:{colour}'>{res['prediction']}</span><br>"
            f"P(still active) = {res['p_active']:.1%} | churn risk = {res['churn_risk']:.1%} (threshold {THRESHOLD:.0%})<br>"
            f"<small>saved to {LOG_PATH}</small></div>"
        ))
        plot_waterfall(pd.DataFrame([{f: inputs[f].value for f in FEATURES}]), f'Why this churn risk: {pid}')


use_default.observe(on_reset, names='value')
button.on_click(on_click)
display(w.VBox(rows + [use_default, button, out]))

## 3. Stored predictions

Run this cell again after saving new players to refresh it.

In [ ]:
df_log = load_log()
if df_log.empty:
    print('no stored prediction yet: use the form above')
else:
    print(len(df_log), 'stored predictions | at risk:', int((df_log['churn_risk'] >= THRESHOLD).sum()))
    display(df_log[['timestamp', 'player_id', 'p_active', 'churn_risk', 'prediction'] + FEATURES].sort_values('timestamp', ascending=False))
    fig, ax = plt.subplots(figsize=(8, max(2, 0.4 * len(df_log))))
    d = df_log.sort_values('churn_risk')
    ax.barh(d['player_id'].astype(str) + ' (' + d['timestamp'].str[11:] + ')', d['churn_risk'], color=np.where(d['churn_risk'] >= THRESHOLD, '#c0392b', '#27ae60'))
    ax.axvline(THRESHOLD, color='black', linestyle='--')
    ax.set_xlim(0, 1)
    ax.set_xlabel('churn risk (dashed = threshold)')
    plt.show()

## 4. Score a whole file

A CSV or parquet file with one row per player and (some of) the model columns, e.g. `player_features.parquet` itself. Set `SCORE_PATH` and run; the result is saved next to the file and appended to the log if `STORE = True`.

In [ ]:
SCORE_PATH: Path | None = None  # e.g. Path('player_features.parquet') or Path('my_players.csv')
STORE: bool = False

if SCORE_PATH is None:
    print('set SCORE_PATH to score a file')
else:
    df_in = pd.read_parquet(SCORE_PATH) if SCORE_PATH.suffix == '.parquet' else pd.read_csv(SCORE_PATH)
    if 'player_id' not in df_in.columns:
        df_in.insert(0, 'player_id', df_in['playerid'] if 'playerid' in df_in.columns else range(len(df_in)))
    df_res = predict_players(df_in[['player_id'] + [f for f in FEATURES if f in df_in.columns]])
    df_res.to_csv(SCORE_PATH.with_name(SCORE_PATH.stem + '_scored.csv'), index=False)
    if STORE:
        save_predictions(df_res)
    print(len(df_res), 'players scored | at risk:', int((df_res['churn_risk'] >= THRESHOLD).sum()))
    display(df_res.sort_values('churn_risk', ascending=False).head(20))

## 5. SHAP: why does the model predict churn?

**Global view**: the SHAP values of the 840 training players (aggregated table only). Each dot is a player; a point to the right raises the churn risk, to the left lowers it; the colour is the feature value (red = high). The bar chart is the mean absolute impact per feature.

In [ ]:
df_all = pd.read_parquet(FEATURES_PATH)
exp_all = shap_churn(df_all)
shap.plots.beeswarm(exp_all, max_display=12, show=False)
plt.title('Impact on churn risk (all training players)', loc='left')
plt.tight_layout()
plt.show()
shap.plots.bar(exp_all, max_display=12, show=False)
plt.title('Mean absolute impact on churn risk', loc='left')
plt.tight_layout()
plt.show()

**One stored prediction**: explains the stored prediction at position `STORED_INDEX` of the log (-1 = the latest). The form of section 2 already draws the same diagram after each click.

In [ ]:
STORED_INDEX: int = -1

df_log = load_log()
if df_log.empty:
    print('no stored prediction yet: use the form above')
else:
    row = df_log.iloc[[STORED_INDEX]]
    plot_waterfall(row, f"Why this churn risk: {row['player_id'].iloc[0]} ({row['timestamp'].iloc[0]})")

## 6. Sanity check (nothing is stored)

Two example players: a regular recent player and a player silent for a long time.

In [ ]:
examples = {
    'regular recent player': dict(days_since_last_match=2, matches_last_30d=40, matches_last_90d=120, matches=300, tenure_days=250, sessions_per_week=5, share_pvp_ranked=0.5, n_modes=3),
    'silent one-timer': dict(days_since_last_match=200, matches_last_30d=0, matches_last_90d=0, matches=3, tenure_days=1, sessions_per_week=1, share_pvp_ranked=0.0, n_modes=1),
}
pd.DataFrame([predict_one(name, vals, store=False) for name, vals in examples.items()])[['player_id', 'p_active', 'churn_risk', 'prediction']]

In [ ]:
plot_waterfall(pd.DataFrame([examples['silent one-timer']]), 'Why this churn risk: silent one-timer')